# Phase 0 — SparseVLM baseline reproduction on Kaggle 2x T4

**Scope:** set up the original LLaVA-1.5-7B checkpoint, install SparseVLM's codebase, and reproduce
its accuracy numbers vs. the vanilla baseline on a POPE subset.

**Run this in a separate Kaggle notebook / separate session from the FastV one** — the two use
different codebases (HF-native `transformers` fork vs. the original `haotian-liu/LLaVA` fork) with
different pinned dependency versions, and installing both in one environment risks version conflicts
I can't fully predict without testing.

**Important hardware caveat I found by reading the code, not guessing:** SparseVLM's model loader
(`llava/model/builder.py`) attaches a `LlamaDynamicvitFlashAttention2` submodule to every layer when
loading the sparse model, and the official `flash-attn` package (v2) **does not support Turing GPUs —
T4 included** (confirmed via the library's own README and other users hitting
`RuntimeError: FlashAttention only supports Ampere GPUs or newer` on Kaggle T4s specifically).
However, the actual attention path used in the forward pass is asserted to be **SDPA**
(`assert config._attn_implementation == 'sdpa'`), which *does* work fine on T4 — the flash-attn
submodule looks like it's attached but not on the critical path for basic greedy-decoding eval.
**This is my best reading of the code, not a tested guarantee** — the smoke test in Section 3 will
tell us quickly whether this holds. If model loading crashes on the flash-attn submodule attachment
specifically (not the SDPA path), Section 3 includes a documented workaround.


## 1. Environment check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory/1e9:.1f} GB, compute capability {props.major}.{props.minor}")
print("Existing torch version (do not upgrade this):", torch.__version__, "| CUDA build:", torch.version.cuda)


## 2. Install SparseVLM

SparseVLM's `pyproject.toml` pins `torch==2.1.2`, `transformers==4.37.2`, `accelerate==0.21.0`.
**Do not let pip upgrade/downgrade torch** — Kaggle's pre-installed torch is already matched to the
GPU driver; reinstalling a different torch build risks breaking CUDA entirely. We install with
`--no-deps` and then add only the non-torch pinned packages explicitly.


In [1]:
%%bash
cd /kaggle/working
if [ ! -d "SparseVLMs" ]; then
  git clone --depth 1 https://github.com/Gumpest/SparseVLMs.git
fi
cd SparseVLMs
pip install -e . --no-deps -q
pip install -q "transformers==4.37.2" "tokenizers==0.15.1" "accelerate==0.21.0" \
    sentencepiece shortuuid peft bitsandbytes einops==0.6.1 einops-exts==0.0.4 timm==0.6.13
python -c "import llava; print('llava package import OK')"
python -c "import transformers; print('transformers version:', transformers.__version__)"


llava package import OK
transformers version: 4.37.2


USE_VERSION: 1_0


## 3. Smoke test — baseline vs. SparseVLM on one image

Loads `liuhaotian/llava-v1.5-7b` twice: once as the vanilla `LlavaLlamaForCausalLM` (baseline), once
as `LlavaLlamaDynamicForCausalLM` (SparseVLM), both in 8-bit. We import the two classes directly
rather than going through `builder.py`'s name-matching dispatch, so we know exactly which model we
get.

**If this cell fails specifically inside the `flash_attn` submodule-attachment loop** (traceback
will mention `LlamaDynamicvitFlashAttention2` or an import of the `flash_attn` package), open
`/kaggle/working/SparseVLMs/llava/model/builder.py` (it's an editable install, so this file is live)
and comment out the `for i in range(32): ...` block that adds the `flash_attn` submodule (right after
`model = LlavaLlamaDynamicForCausalLM.from_pretrained(...)`), then re-run this cell. Based on the
`assert config._attn_implementation == 'sdpa'` in `modelling_sparse_llama.py`, that block shouldn't
be needed for the forward path this eval exercises — but I flag this as something to verify, not
something I've confirmed.


In [2]:
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig
from PIL import Image
import requests

MODEL_PATH = "liuhaotian/llava-v1.5-7b"
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

from llava.model.language_model.llava_llama import LlavaLlamaForCausalLM
from llava.model.language_model.sparse_llava_llama import LlavaLlamaDynamicForCausalLM
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
from llava.conversation import conv_templates
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
image = Image.open(requests.get("http://images.cocodataset.org/val2017/000000039769.jpg", stream=True).raw).convert("RGB")

def build_prompt(question, model_config):
    qs = question
    if model_config.mm_use_im_start_end:
        qs = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + qs
    else:
        qs = DEFAULT_IMAGE_TOKEN + '\n' + qs
    conv = conv_templates["vicuna_v1"].copy()
    conv.append_message(conv.roles[0], qs)
    conv.append_message(conv.roles[1], None)
    return conv.get_prompt()

def run_once(model_cls, tag):
    print(f"Loading {tag}...")
    model = model_cls.from_pretrained(
        MODEL_PATH, low_cpu_mem_usage=True,
        quantization_config=bnb_config, attn_implementation="sdpa",
    )
    vision_tower = model.get_vision_tower()
    if not vision_tower.is_loaded:
        vision_tower.load_model(device_map=model.device)
    image_processor = vision_tower.image_processor
    model.config.mm_use_im_start_end = getattr(model.config, "mm_use_im_start_end", False)

    prompt = build_prompt("What is shown in this image? Answer in one sentence.", model.config)
    input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).to(model.device)
    image_tensor = process_images([image], image_processor, model.config).to(model.device, dtype=torch.float16)

    with torch.inference_mode():
        out = model.generate(input_ids, images=image_tensor, image_sizes=[image.size],
                              do_sample=False, max_new_tokens=64, use_cache=True)
    text = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    print(f"{tag}:", text)
    del model
    torch.cuda.empty_cache()
    return text

baseline_out = run_once(LlavaLlamaForCausalLM, "BASELINE")
sparse_out = run_once(LlavaLlamaDynamicForCausalLM, "SPARSEVLM")


USE_VERSION: 1_0
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Loading BASELINE...


...

You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.


BASELINE: Two cats are laying on a pink blanket.
Loading SPARSEVLM...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 398.77, TFLOPs:1.71
                  Total Time (ms): 2307.06


SPARSEVLM: Two cats are laying on a pink couch.


**Checkpoint:** same as the FastV notebook — both outputs should be coherent. Paste back the outputs
and/or any traceback before continuing to the full POPE run below.

## 4. POPE — data prep (same subset approach as the FastV notebook)

If you already ran the FastV notebook's data-prep cell, you can skip re-downloading by pointing
`img_dir`/`pope_dir` at the same paths (Kaggle notebooks don't share `/kaggle/working` across
separate sessions by default, though, so re-running this is the safe default).

In [3]:
%%bash
mkdir -p /kaggle/working/data
cd /kaggle/working/data
if [ ! -d "POPE" ]; then
  git clone --depth 1 https://github.com/AoiDragon/POPE.git
fi


Cloning into 'POPE'...


In [4]:
import json, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import urllib.request

random.seed(0)
N_PER_SPLIT = 150

pope_dir = Path("/kaggle/working/data/POPE/output/coco")
splits = ["coco_pope_random.json", "coco_pope_popular.json", "coco_pope_adversarial.json"]

subset = {}
for split in splits:
    lines = [json.loads(l) for l in open(pope_dir / split) if l.strip()]
    images = sorted(set(l["image"] for l in lines))
    random.shuffle(images)
    keep_images = set(images[: max(1, N_PER_SPLIT // 6)])
    sampled = [l for l in lines if l["image"] in keep_images][:N_PER_SPLIT]
    subset[split] = sampled
    print(split, "->", len(sampled), "questions")

img_dir = Path("/kaggle/working/data/coco_val2014_subset")
img_dir.mkdir(exist_ok=True)
all_images = sorted(set(l["image"] for s in subset.values() for l in s))

def fetch(fname):
    dest = img_dir / fname
    if dest.exists():
        return
    urllib.request.urlretrieve(f"http://images.cocodataset.org/val2014/{fname}", dest)

with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(fetch, all_images))
print("Downloaded:", len(list(img_dir.glob('*.jpg'))), "images")


coco_pope_random.json -> 150 questions
coco_pope_popular.json -> 150 questions
coco_pope_adversarial.json -> 150 questions
Downloaded: 73 images


## 5. POPE — run baseline + SparseVLM inference

In [5]:
import re, gc, torch
from PIL import Image

all_questions = [l for s in subset.values() for l in s]
print("Total POPE questions:", len(all_questions))

def parse_yes_no(text):
    text = text.strip().lower()
    if text.startswith("yes"):
        return "yes"
    if text.startswith("no"):
        return "no"
    m = re.search(r"\b(yes|no)\b", text)
    return m.group(1) if m else "no"

def run_eval(model_cls, tag):
    model = model_cls.from_pretrained(
        MODEL_PATH, low_cpu_mem_usage=True,
        quantization_config=bnb_config, attn_implementation="sdpa",
    )
    vision_tower = model.get_vision_tower()
    if not vision_tower.is_loaded:
        vision_tower.load_model(device_map=model.device)
    image_processor = vision_tower.image_processor
    model.config.mm_use_im_start_end = getattr(model.config, "mm_use_im_start_end", False)
    model.eval()

    preds, labels = [], []
    for i, q in enumerate(all_questions):
        img = Image.open(img_dir / q["image"]).convert("RGB")
        prompt = build_prompt(q["text"], model.config)
        input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).to(model.device)
        image_tensor = process_images([img], image_processor, model.config).to(model.device, dtype=torch.float16)
        with torch.inference_mode():
            out = model.generate(input_ids, images=image_tensor, image_sizes=[img.size],
                                  do_sample=False, max_new_tokens=10, use_cache=True)
        text = tokenizer.decode(out[0], skip_special_tokens=True)
        preds.append(parse_yes_no(text))
        labels.append(q["label"])
        if (i + 1) % 50 == 0:
            print(f"  [{tag}] {i+1}/{len(all_questions)}")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return preds, labels

def pope_metrics(preds, labels):
    tp = sum(p == "yes" and l == "yes" for p, l in zip(preds, labels))
    fp = sum(p == "yes" and l == "no" for p, l in zip(preds, labels))
    tn = sum(p == "no" and l == "no" for p, l in zip(preds, labels))
    fn = sum(p == "no" and l == "yes" for p, l in zip(preds, labels))
    acc = (tp + tn) / len(labels)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1, yes_ratio=preds.count("yes")/len(preds))

results = {}
print("=== Baseline ===")
preds, labels = run_eval(LlavaLlamaForCausalLM, "baseline")
results["baseline"] = pope_metrics(preds, labels)

print("=== SparseVLM ===")
preds, labels = run_eval(LlavaLlamaDynamicForCausalLM, "sparsevlm")
results["sparsevlm"] = pope_metrics(preds, labels)

print("\n==== POPE RESULTS SUMMARY ====")
print(f"{'setting':<12}{'acc':>8}{'f1':>8}{'yes%':>8}")
for tag, m in results.items():
    print(f"{tag:<12}{m['accuracy']*100:>7.1f}%{m['f1']*100:>7.1f}%{m['yes_ratio']*100:>7.1f}%")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.


Total POPE questions: 450
=== Baseline ===


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  [baseline] 50/450
  [baseline] 100/450
  [baseline] 150/450
  [baseline] 200/450
  [baseline] 250/450
  [baseline] 300/450
  [baseline] 350/450
  [baseline] 400/450
  [baseline] 450/450
=== SparseVLM ===


You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.


...

  [sparsevlm] 350/450


[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 90925.99, TFLOPs:1.69
                  Total Time (ms): 554569.05
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 91180.81, TFLOPs:1.69
                  Total Time (ms): 556130.56
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 91435.97, TFLOPs:1.69
                  Total Time (ms): 557694.77
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 91687.75, TFLOPs:1.69
                  Total Time (ms): 559207.79
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 91974.92, TFLOPs:1.69
                  Total Time (ms): 560796.10
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 92232.26, TFLOPs:1.69
                  Total Time (ms): 562353.43
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 92491.82, TFLOPs:1.69
                  Total Time (ms): 563916.96
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 92752.24, TFLOPs:1.69
                  Total Time (ms): 565474.53
[Efficiency INFO

  [sparsevlm] 400/450


[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 103819.16, TFLOPs:1.69
                  Total Time (ms): 632534.44
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 104099.72, TFLOPs:1.69
                  Total Time (ms): 634135.39
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 104360.50, TFLOPs:1.69
                  Total Time (ms): 635711.19
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 104609.32, TFLOPs:1.69
                  Total Time (ms): 637201.55
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 104863.69, TFLOPs:1.69
                  Total Time (ms): 638770.68
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 105116.69, TFLOPs:1.69
                  Total Time (ms): 640282.43
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 105371.68, TFLOPs:1.69
                  Total Time (ms): 641851.32
[Efficiency INFO] Equal Tokens: 192, Prefill Time (ms): 105643.37, TFLOPs:1.69
                  Total Time (ms): 643403.69
[Efficie

  [sparsevlm] 450/450

==== POPE RESULTS SUMMARY ====
setting          acc      f1    yes%
baseline       87.8%   88.2%   53.3%
sparsevlm      87.6%   87.8%   52.2%


## Summary — what to send back

1. Section 3 smoke-test outputs (and whether you needed the flash-attn workaround)
2. Section 5's `==== POPE RESULTS SUMMARY ====` table
3. Any traceback, in full

Once both this and the FastV notebook produce sane numbers, we've de-risked the environment for
Phase 1 (porting the InfoScore entropy criterion in).